In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import pickle


In [2]:
df=pd.read_csv('../data/processed/processed_data.csv')

In [3]:
df = df.sort_values(['product_id', 'date']).reset_index(drop=True)

In [4]:
df['lag_7'] = df.groupby('product_id')['units_sold'].shift(7)

In [5]:
df['lag_14'] = df.groupby('product_id')['units_sold'].shift(14)

In [6]:
df['rolling_mean_7'] = df.groupby('product_id')['units_sold'].rolling(window=7).mean().reset_index(level=0, drop=True)
df['rolling_std_7'] = df.groupby('product_id')['units_sold'].rolling(window=7).std().reset_index(level=0, drop=True)

In [7]:
df['rolling_mean_30'] = df.groupby('product_id')['units_sold'].rolling(window=30).mean().reset_index(level=0, drop=True)
df['rolling_std_30'] = df.groupby('product_id')['units_sold'].rolling(window=30).std().reset_index(level=0, drop=True)

In [8]:
df['date'] = pd.to_datetime(df['date'])
df['is_weekend'] = np.where(df['day_of_week'].isin([5,6]), 1, 0)

In [9]:
df["quarter"] = df["date"].dt.quarter
    

In [10]:
df["quarter_sin"] = np.sin(2 * np.pi * df["quarter"] / 4)
df["quarter_cos"] = np.cos(2 * np.pi * df["quarter"] / 4)

In [11]:
target_cols = [
    "lag_7",
    "lag_14",
    "rolling_mean_7",
    "rolling_std_7",
    "rolling_mean_30",
    "rolling_std_30",
]

In [12]:
df = df.dropna(subset=target_cols)

In [13]:
split_index = int(len(df) * 0.8)
train = df.iloc[:split_index].copy()
test = df.iloc[split_index:].copy()

In [14]:
cols_to_scale=['units_sold',
'unit_price',
'stock_on_hand',
'reorder_point',
'discount_pct',
'supplier_lead_days',
'lag_7', 'lag_14',
'rolling_mean_7', 'rolling_std_7',
'rolling_mean_30', 'rolling_std_30']

In [15]:
scaler= MinMaxScaler()
train[cols_to_scale] = scaler.fit_transform(train[cols_to_scale])
test[cols_to_scale] = scaler.transform(test[cols_to_scale])

In [16]:
with open('../models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

In [17]:

train.to_csv('../data/processed/train_engineered.csv', index=False)
test.to_csv('../data/processed/test_engineered.csv', index=False)